# P3.09 Discovery: Output Retrieval Probe

**PURPOSE:** Test artifact retrieval, metadata validation, and checksum verification after execution

**TIMEOUT:** ≤5 minutes

**CRITICAL:** P3.09 must retrieve shard outputs reliably with integrity validation.

In [ ]:
import json
import time
import hashlib
from datetime import datetime
from pathlib import Path

DISCOVERY_SESSION = {
    "session_id": f"output_probe_{int(time.time())}",
    "timestamp": datetime.now().isoformat(),
    "timeout_hard": 300,
    "timeout_warning": 270,
    "notebook_name": "kaggle_discovery_08_output_retrieval_probe",
    "results": []
}

start_time = time.time()

def log_test(test_name, result, evidence, duration_s):
    DISCOVERY_SESSION["results"].append({
        "test": test_name,
        "result": result,
        "evidence": evidence,
        "duration_s": duration_s,
        "timestamp": datetime.now().isoformat()
    })
    print(f"[{result:8s}] {test_name} ({duration_s:.1f}s)")

def check_timeout():
    elapsed = time.time() - start_time
    if elapsed > DISCOVERY_SESSION["timeout_hard"]:
        raise RuntimeError(f"HARD TIMEOUT: {elapsed:.0f}s")
    return elapsed

print(f"🔬 P3.09 Output Retrieval Probe: {DISCOVERY_SESSION['session_id']}")
print(f"📍 Started: {DISCOVERY_SESSION['timestamp']}")

## Test 1: Artifact Storage in /kaggle/working/

In [ ]:
test_start = time.time()
check_timeout()

try:
    storage_probe = {
        "working_dir_accessible": False,
        "test_file_stored": False,
        "storage_path": "/kaggle/working/test_artifact.json"
    }
    
    working_dir = Path("/kaggle/working")
    storage_probe["working_dir_accessible"] = working_dir.exists() and working_dir.is_dir()
    
    if storage_probe["working_dir_accessible"]:
        # Store a test artifact
        artifact_data = {
            "test_id": "output_probe_test",
            "timestamp": datetime.now().isoformat(),
            "content": "test artifact data"
        }
        artifact_file = working_dir / "test_artifact.json"
        artifact_file.write_text(json.dumps(artifact_data, indent=2))
        storage_probe["test_file_stored"] = artifact_file.exists()
        
        if storage_probe["test_file_stored"]:
            storage_probe["file_size_bytes"] = artifact_file.stat().st_size
            # Don't clean up yet; we'll retrieve it in next test
    
    result_status = "PASS" if storage_probe["test_file_stored"] else "FAIL"
    duration = time.time() - test_start
    log_test("artifact_storage", result_status, storage_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("artifact_storage", "FAIL", str(e)[:100], duration)

## Test 2: Output File Metadata Retrieval

In [ ]:
test_start = time.time()
check_timeout()

try:
    metadata_probe = {
        "artifact_found": False,
        "metadata": {}
    }
    
    artifact_file = Path("/kaggle/working/test_artifact.json")
    
    if artifact_file.exists():
        metadata_probe["artifact_found"] = True
        stat = artifact_file.stat()
        metadata_probe["metadata"] = {
            "size_bytes": stat.st_size,
            "modified_time": str(datetime.fromtimestamp(stat.st_mtime)),
            "is_file": artifact_file.is_file(),
            "readable": artifact_file.is_file()
        }
    
    result_status = "PASS" if metadata_probe["artifact_found"] else "FAIL"
    duration = time.time() - test_start
    log_test("output_metadata_retrieval", result_status, metadata_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("output_metadata_retrieval", "FAIL", str(e)[:100], duration)

## Test 3: Checksum/Hash Validation

In [ ]:
test_start = time.time()
check_timeout()

try:
    checksum_probe = {
        "hash_computed": False,
        "hash_algorithm": "SHA256",
        "hash_value": None,
        "hash_verification": False
    }
    
    artifact_file = Path("/kaggle/working/test_artifact.json")
    
    if artifact_file.exists():
        # Compute SHA256 hash
        sha256_hash = hashlib.sha256()
        with open(artifact_file, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256_hash.update(chunk)
        
        checksum_probe["hash_computed"] = True
        checksum_probe["hash_value"] = sha256_hash.hexdigest()
        
        # Verify by re-reading
        sha256_verify = hashlib.sha256()
        with open(artifact_file, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256_verify.update(chunk)
        
        checksum_probe["hash_verification"] = sha256_hash.hexdigest() == sha256_verify.hexdigest()
    
    result_status = "PASS" if checksum_probe["hash_verification"] else "FAIL"
    duration = time.time() - test_start
    log_test("checksum_validation", result_status, checksum_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("checksum_validation", "FAIL", str(e)[:100], duration)

## Test 4: Output Retrievability & Cleanup

In [ ]:
test_start = time.time()
check_timeout()

try:
    retrieve_probe = {
        "file_retrieved": False,
        "content_matches": False,
        "cleanup_successful": False
    }
    
    artifact_file = Path("/kaggle/working/test_artifact.json")
    
    if artifact_file.exists():
        # Retrieve and parse content
        try:
            content = artifact_file.read_text()
            data = json.loads(content)
            retrieve_probe["file_retrieved"] = True
            retrieve_probe["content_matches"] = data.get("test_id") == "output_probe_test"
        except:
            pass
        
        # Clean up
        try:
            artifact_file.unlink()
            retrieve_probe["cleanup_successful"] = not artifact_file.exists()
        except:
            pass
    
    result_status = "PASS" if retrieve_probe["content_matches"] else "FAIL"
    duration = time.time() - test_start
    log_test("output_retrievability", result_status, retrieve_probe, duration)
except Exception as e:
    duration = time.time() - test_start
    log_test("output_retrievability", "FAIL", str(e)[:100], duration)

## Final Report

In [ ]:
results = DISCOVERY_SESSION["results"]
passed = sum(1 for r in results if r["result"] == "PASS")
failed = sum(1 for r in results if r["result"] == "FAIL")
unknown = sum(1 for r in results if r["result"] == "UNKNOWN")

total_time = time.time() - start_time

DISCOVERY_SESSION.update({
    "summary": {
        "total_tests": len(results),
        "passed": passed,
        "failed": failed,
        "unknown": unknown,
        "total_time_s": total_time,
        "verdict": "Output retrieval validated" if passed >= 3 else "Output retrieval needs investigation"
    }
})

output_path = Path("/kaggle/working/discovery_output_probe_results.json")
output_path.write_text(json.dumps(DISCOVERY_SESSION, indent=2))

print(f"\n📊 Output Retrieval Summary:")
print(f"   PASS:    {passed}")
print(f"   FAIL:    {failed}")
print(f"   UNKNOWN: {unknown}")
print(f"   Total:   {len(results)} tests in {total_time:.1f}s")
print(f"\n✅ Results saved to: {output_path}")